# Clasificación de compuertas lógicas dibujadas a mano

**PC2 · Computación Gráfica · UNI**

**Problema:** digitalizar circuitos lógicos dibujados a mano. Como primer paso, una red neuronal debe reconocer qué compuerta se dibujó.

**Clases (6):** AND, OR, NOT, NAND, NOR, XOR.

**Datos:** dibujos recolectados por el equipo con la app web del repositorio (lienzo de 256×256). Cada PNG está en una carpeta con el nombre de su clase y el trazo va en el canal alfa.

**Contenido del notebook**
1. Carga de datos
2. Preprocesamiento
3. Exploración del dataset
4. División en entrenamiento, validación y prueba
5. Modelo base: perceptrón multicapa (MLP)
6. Red neuronal convolucional (CNN)
7. Métricas: accuracy, precisión, recall, F1 y matriz de confusión
8. Análisis de errores y parejas difíciles (AND/NAND, OR/NOR, OR/XOR)
9. Experimento: ¿cuánto importa la resolución de la imagen?
10. Generalización a personas que no participaron en el entrenamiento
11. Conclusiones

## 1. Carga de datos

Hay tres formas de cargar el dataset (se usa la primera que aplique):
- **`URL_DATASET`**: la dirección de la app desplegada, por ejemplo `https://faalvial.pythonanywhere.com/dataset.zip`.
- **`RUTA_ZIP`**: un `.zip` que ya está en la máquina (por ejemplo, subido al panel de archivos de Colab).
- Si ninguna existe y estamos en Colab, se abre un diálogo para subir el `.zip`.

In [ ]:
import os, io, json, zipfile, urllib.request, collections
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

URL_DATASET = os.environ.get("URL_DATASET", "")    # ej. "https://faalvial.pythonanywhere.com/dataset.zip"
RUTA_ZIP = os.environ.get("RUTA_ZIP", "dataset.zip")
CARPETA = "dataset"
CLASES = ["AND", "OR", "NOT", "NAND", "NOR", "XOR"]

if URL_DATASET:
    print("Descargando", URL_DATASET)
    urllib.request.urlretrieve(URL_DATASET, RUTA_ZIP)
elif not os.path.exists(RUTA_ZIP):
    from google.colab import files  # solo existe en Colab
    subido = files.upload()
    RUTA_ZIP = next(iter(subido))

with zipfile.ZipFile(RUTA_ZIP) as zf:
    zf.extractall(CARPETA)
print("Dataset extraído en", CARPETA)

## 2. Preprocesamiento

Cada dibujo pasa por estos pasos:
1. **Canal alfa:** el lienzo es transparente y el trazo es opaco, así que el canal alfa ya es la imagen en escala de grises (0 = fondo, 255 = tinta).
2. **Recorte:** se recorta al rectángulo que contiene la tinta, con un margen. Así no importa en qué parte del lienzo ni de qué tamaño dibujó cada persona.
3. **Cuadrado:** se rellena con fondo para que quede cuadrado sin deformar la compuerta.
4. **Redimensionar:** a `TAM × TAM` píxeles (64 por defecto) y normalizar a valores entre 0 y 1.

Los recortes cuadrados se guardan a resolución completa para poder generar luego versiones de 16, 28 o 64 píxeles en el experimento de resolución.

In [ ]:
TAM = 64
MARGEN = 0.08

def recortar_cuadrado(ruta, margen=MARGEN):
    alfa = Image.open(ruta).convert("RGBA").getchannel("A")
    caja = alfa.getbbox()                     # rectángulo con tinta
    if caja is None:
        return None
    x0, y0, x1, y1 = caja
    lado = int(max(x1 - x0, y1 - y0) * (1 + 2 * margen))
    cuadrado = Image.new("L", (lado, lado), 0)
    cuadrado.paste(alfa.crop(caja), ((lado - (x1 - x0)) // 2, (lado - (y1 - y0)) // 2))
    return cuadrado

def a_matriz(recortes, tam):
    return np.stack([np.asarray(r.resize((tam, tam), Image.Resampling.LANCZOS), dtype=np.float32) / 255.0
                     for r in recortes])

recortes, etiquetas, autores, archivos = [], [], [], []
for i, clase in enumerate(CLASES):
    carpeta = os.path.join(CARPETA, clase)
    for nombre in sorted(os.listdir(carpeta)):
        if not nombre.endswith(".png"):
            continue
        r = recortar_cuadrado(os.path.join(carpeta, nombre))
        if r is None:
            continue
        recortes.append(r)
        etiquetas.append(i)
        partes = nombre.split("_")
        autores.append(partes[1] if len(partes) >= 3 else "anonimo")
        archivos.append(f"{clase}/{nombre}")

y = np.array(etiquetas)
autores = np.array(autores)
X = a_matriz(recortes, TAM)
print("Imágenes:", X.shape, " Etiquetas:", y.shape)

## 3. Exploración del dataset

In [ ]:
conteo = np.bincount(y, minlength=len(CLASES))
fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
ax[0].bar(CLASES, conteo, color="#8a1c2b")
ax[0].set_title("Dibujos por compuerta")
for i, n in enumerate(conteo):
    ax[0].text(i, n, str(n), ha="center", va="bottom")
por_autor = collections.Counter(autores)
nombres = [a for a, _ in por_autor.most_common()]
ax[1].bar(nombres, [por_autor[a] for a in nombres], color="#5a626c")
ax[1].set_title("Dibujos por persona")
ax[1].tick_params(axis="x", rotation=45)
plt.tight_layout(); plt.show()
print(dict(zip(CLASES, conteo)), " total:", conteo.sum())

In [ ]:
rng = np.random.default_rng(0)
fig, axes = plt.subplots(len(CLASES), 8, figsize=(10, 8))
for i, clase in enumerate(CLASES):
    idx = rng.choice(np.where(y == i)[0], size=min(8, (y == i).sum()), replace=False)
    for j in range(8):
        a = axes[i, j]; a.axis("off")
        if j < len(idx):
            a.imshow(X[idx[j]], cmap="gray_r")
    axes[i, 0].set_title(clase, loc="left", fontsize=10)
plt.suptitle(f"Ejemplos después del preprocesamiento ({TAM}×{TAM})"); plt.tight_layout(); plt.show()

La **imagen promedio** de cada clase muestra qué tan consistentes son los dibujos. En NAND y NOR debería verse el círculo de la salida; en XOR, la curva extra.

In [ ]:
fig, axes = plt.subplots(1, len(CLASES), figsize=(12, 2.4))
for i, clase in enumerate(CLASES):
    axes[i].imshow(X[y == i].mean(axis=0), cmap="magma"); axes[i].set_title(clase); axes[i].axis("off")
plt.suptitle("Imagen promedio por compuerta"); plt.tight_layout(); plt.show()

## 4. División en entrenamiento, validación y prueba

- **Entrenamiento (70%)**: la red aprende con estos datos.
- **Validación (15%)**: sirve para decidir cuándo dejar de entrenar.
- **Prueba (15%)**: solo se usa al final para medir el rendimiento real.

La división es **estratificada** (cada conjunto tiene la misma proporción de cada compuerta) y se guardan los índices para reutilizar exactamente la misma división en todos los experimentos.

In [ ]:
from sklearn.model_selection import train_test_split
import logging
import keras
from keras import layers

logging.getLogger("tensorflow").setLevel(logging.ERROR)  # oculta avisos internos de TensorFlow

SEMILLA = 42
keras.utils.set_random_seed(SEMILLA)

indices = np.arange(len(y))
idx_train, idx_tmp = train_test_split(indices, test_size=0.30, stratify=y, random_state=SEMILLA)
idx_val, idx_test = train_test_split(idx_tmp, test_size=0.50, stratify=y[idx_tmp], random_state=SEMILLA)

def separar(Xs):
    Xs = Xs[..., None]  # canal de color
    return Xs[idx_train], Xs[idx_val], Xs[idx_test]

X_train, X_val, X_test = separar(X)
y_train, y_val, y_test = y[idx_train], y[idx_val], y[idx_test]
print("Entrenamiento:", X_train.shape, " Validación:", X_val.shape, " Prueba:", X_test.shape)

## 5. Modelo base: perceptrón multicapa (MLP)

Primero una red densa simple, que ve la imagen como un vector de píxeles sin estructura espacial. Sirve como punto de comparación para la CNN.

In [ ]:
EPOCAS = int(os.environ.get("EPOCAS", 60))

def callbacks():
    return [keras.callbacks.EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True),
            keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4)]

def crear_mlp(tam):
    return keras.Sequential([
        keras.Input((tam, tam, 1)),
        layers.Flatten(),
        layers.Dense(256, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(len(CLASES), activation="softmax"),
    ], name="MLP")

mlp = crear_mlp(TAM)
mlp.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
mlp.summary()
hist_mlp = mlp.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=EPOCAS,
                   batch_size=64, callbacks=callbacks(), verbose=2)

## 6. Red neuronal convolucional (CNN)

La CNN aprende filtros que detectan bordes, curvas y círculos, y luego los combina. Eso la hace adecuada para distinguir detalles como el círculo de NAND o la curva extra de XOR.

**Aumento de datos:** durante el entrenamiento cada imagen se rota (±10°), se desplaza y se acerca o aleja un poco al azar, para simular la variación natural entre personas. **No** se aplica volteo horizontal: una compuerta volteada tendría la salida a la izquierda, que no es un dibujo válido.

**Arquitectura:** tres bloques de convolución + max pooling (32, 64 y 128 filtros), seguidos de una capa densa con dropout para reducir el sobreajuste.

In [ ]:
def crear_aumento():
    return keras.Sequential([
        layers.RandomRotation(0.03, fill_mode="constant"),
        layers.RandomTranslation(0.08, 0.08, fill_mode="constant"),
        layers.RandomZoom(0.1, fill_mode="constant"),
    ], name="aumento_de_datos")

def crear_cnn(tam):
    return keras.Sequential([
        keras.Input((tam, tam, 1)),
        crear_aumento(),
        layers.Conv2D(32, 3, padding="same", activation="relu"),
        layers.Conv2D(32, 3, padding="same", activation="relu"),
        layers.MaxPooling2D(),
        layers.Conv2D(64, 3, padding="same", activation="relu"),
        layers.Conv2D(64, 3, padding="same", activation="relu"),
        layers.MaxPooling2D(),
        layers.Conv2D(128, 3, padding="same", activation="relu"),
        layers.MaxPooling2D(),
        layers.Flatten(),
        layers.Dropout(0.4),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(len(CLASES), activation="softmax"),
    ], name="CNN")

cnn = crear_cnn(TAM)
cnn.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
cnn.summary()
hist_cnn = cnn.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=EPOCAS,
                   batch_size=64, callbacks=callbacks(), verbose=2)

### Curvas de entrenamiento

Si la pérdida de validación sube mientras la de entrenamiento baja, la red está memorizando (sobreajuste). `EarlyStopping` detiene el entrenamiento en ese punto y recupera los mejores pesos.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
for h, nombre, color in [(hist_mlp, "MLP", "#5a626c"), (hist_cnn, "CNN", "#8a1c2b")]:
    axes[0].plot(h.history["loss"], color=color, label=f"{nombre} entrenamiento")
    axes[0].plot(h.history["val_loss"], color=color, ls="--", label=f"{nombre} validación")
    axes[1].plot(h.history["accuracy"], color=color, label=f"{nombre} entrenamiento")
    axes[1].plot(h.history["val_accuracy"], color=color, ls="--", label=f"{nombre} validación")
axes[0].set_title("Pérdida"); axes[1].set_title("Accuracy")
for a in axes:
    a.set_xlabel("Época"); a.legend(fontsize=8); a.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 7. Métricas en el conjunto de prueba

- **Accuracy:** porcentaje de dibujos clasificados correctamente.
- **Precisión** de una clase: de los dibujos que la red dijo que eran NAND, cuántos lo eran de verdad.
- **Recall** de una clase: de los NAND reales, cuántos encontró la red.
- **F1:** media armónica de precisión y recall.
- **Matriz de confusión:** fila = compuerta real, columna = compuerta predicha. Lo que está fuera de la diagonal son errores.

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, ConfusionMatrixDisplay

def evaluar(modelo, Xt, yt):
    prob = modelo.predict(Xt, verbose=0)
    pred = prob.argmax(axis=1)
    return pred, prob

pred_mlp, _ = evaluar(mlp, X_test, y_test)
pred_cnn, prob_cnn = evaluar(cnn, X_test, y_test)

for nombre, pred in [("MLP", pred_mlp), ("CNN", pred_cnn)]:
    print(f"===== {nombre} =====")
    print(classification_report(y_test, pred, target_names=CLASES, digits=3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
cm = confusion_matrix(y_test, pred_cnn)
ConfusionMatrixDisplay(cm, display_labels=CLASES).plot(ax=axes[0], cmap="Reds", colorbar=False)
axes[0].set_title("CNN: matriz de confusión (cantidades)")
cm_norm = confusion_matrix(y_test, pred_cnn, normalize="true")
ConfusionMatrixDisplay(cm_norm, display_labels=CLASES).plot(ax=axes[1], cmap="Reds", colorbar=False, values_format=".2f")
axes[1].set_title("CNN: matriz de confusión (por fila)")
for a in axes:
    a.set_xlabel("Predicha"); a.set_ylabel("Real")
plt.tight_layout(); plt.show()

## 8. Parejas difíciles y análisis de errores

Estas parejas solo se diferencian por un detalle pequeño. Se mide qué porcentaje de los dibujos de una compuerta se confundió con la otra (en ambos sentidos).

In [ ]:
PAREJAS = [("AND", "NAND"), ("OR", "NOR"), ("OR", "XOR"), ("NOR", "XOR")]

def confusion_parejas(y_real, y_pred):
    cm = confusion_matrix(y_real, y_pred, labels=range(len(CLASES)))
    filas = []
    for a, b in PAREJAS:
        ia, ib = CLASES.index(a), CLASES.index(b)
        filas.append((f"{a} → {b}", 100 * cm[ia, ib] / max(1, cm[ia].sum())))
        filas.append((f"{b} → {a}", 100 * cm[ib, ia] / max(1, cm[ib].sum())))
    return filas

print(f"{'Confusión':<14}{'MLP':>8}{'CNN':>8}")
for (texto, p_mlp), (_, p_cnn) in zip(confusion_parejas(y_test, pred_mlp), confusion_parejas(y_test, pred_cnn)):
    print(f"{texto:<14}{p_mlp:>7.1f}%{p_cnn:>7.1f}%")

### Dibujos que la CNN clasificó mal

In [ ]:
errores = np.where(pred_cnn != y_test)[0]
print(f"{len(errores)} errores de {len(y_test)} dibujos de prueba")
n = min(18, len(errores))
if n:
    cols = 6; filas = int(np.ceil(n / cols))
    fig, axes = plt.subplots(filas, cols, figsize=(12, 2.3 * filas), squeeze=False)
    for k, a in enumerate(axes.flat):
        a.axis("off")
        if k < n:
            i = errores[k]
            a.imshow(X_test[i, ..., 0], cmap="gray_r")
            a.set_title(f"real {CLASES[y_test[i]]}\npred {CLASES[pred_cnn[i]]} ({prob_cnn[i].max():.0%})", fontsize=9)
    plt.tight_layout(); plt.show()

### Comparación MLP contra CNN

In [ ]:
resumen = []
for nombre, modelo, pred in [("MLP", mlp, pred_mlp), ("CNN", cnn, pred_cnn)]:
    resumen.append((nombre, accuracy_score(y_test, pred), f1_score(y_test, pred, average="macro"), modelo.count_params()))
print(f"{'Modelo':<8}{'Accuracy':>10}{'F1 macro':>10}{'Parámetros':>13}")
for nombre, acc, f1, params in resumen:
    print(f"{nombre:<8}{acc:>10.3f}{f1:>10.3f}{params:>13,}")

## 9. Experimento: ¿cuánto importa la resolución?

MNIST usa imágenes de 28×28. ¿Alcanza esa resolución para ver el círculo de NAND y NOR? Se entrena la **misma CNN** con la **misma división de datos** a 16×16, 28×28 y 64×64, y se compara el resultado.

Hipótesis: a baja resolución el círculo se reduce a uno o dos píxeles, así que deberían aumentar las confusiones AND↔NAND y OR↔NOR.

In [ ]:
RESOLUCIONES = [16, 28, 64]
resultados_res = {}
for tam in RESOLUCIONES:
    keras.utils.set_random_seed(SEMILLA)
    Xr_train, Xr_val, Xr_test = separar(a_matriz(recortes, tam))
    modelo = crear_cnn(tam)
    modelo.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    modelo.fit(Xr_train, y_train, validation_data=(Xr_val, y_val), epochs=EPOCAS,
               batch_size=64, callbacks=callbacks(), verbose=0)
    pred = modelo.predict(Xr_test, verbose=0).argmax(axis=1)
    parejas = dict(confusion_parejas(y_test, pred))
    resultados_res[tam] = {
        "accuracy": accuracy_score(y_test, pred),
        "f1": f1_score(y_test, pred, average="macro"),
        "and_nand": (parejas["AND → NAND"] + parejas["NAND → AND"]) / 2,
        "or_nor": (parejas["OR → NOR"] + parejas["NOR → OR"]) / 2,
        "ejemplo": Xr_test[np.where(y_test == CLASES.index("NAND"))[0][0], ..., 0],
    }
    print(f"{tam}×{tam}: accuracy {resultados_res[tam]['accuracy']:.3f}")

In [ ]:
print(f"{'Resolución':<12}{'Accuracy':>10}{'F1 macro':>10}{'AND↔NAND':>11}{'OR↔NOR':>9}")
for tam, r in resultados_res.items():
    print(f"{f'{tam}×{tam}':<12}{r['accuracy']:>10.3f}{r['f1']:>10.3f}{r['and_nand']:>10.1f}%{r['or_nor']:>8.1f}%")

fig, axes = plt.subplots(1, len(RESOLUCIONES) + 1, figsize=(13, 3), gridspec_kw={"width_ratios": [1] * len(RESOLUCIONES) + [2]})
for a, tam in zip(axes, RESOLUCIONES):
    a.imshow(resultados_res[tam]["ejemplo"], cmap="gray_r"); a.set_title(f"NAND a {tam}×{tam}"); a.axis("off")
axes[-1].bar([f"{t}×{t}" for t in RESOLUCIONES], [100 * resultados_res[t]["accuracy"] for t in RESOLUCIONES], color="#8a1c2b")
axes[-1].set_ylabel("Accuracy (%)"); axes[-1].set_ylim(0, 100); axes[-1].set_title("Accuracy según resolución")
plt.tight_layout(); plt.show()

## 10. Generalización a personas nuevas

En la división anterior, los dibujos de una misma persona aparecen tanto en entrenamiento como en prueba, así que la red puede aprender el estilo de cada uno. Una prueba más exigente es **dejar fuera a una persona completa**: entrenar con las demás y evaluar solo con sus dibujos. Se repite para cada persona (validación cruzada por grupos).

Se necesitan al menos 3 personas con nombre. Los dibujos marcados como `anonimo` se usan solo para entrenar.

In [ ]:
personas = [a for a, n in collections.Counter(autores).most_common() if a != "anonimo" and n >= 30]
if len(personas) < 3:
    print("Se necesitan al menos 3 personas con 30 dibujos o más. Personas encontradas:", personas)
else:
    resultados_personas = {}
    errores_personas = collections.Counter()
    for persona in personas:
        keras.utils.set_random_seed(SEMILLA)
        prueba = autores == persona
        idx_resto = np.where(~prueba)[0]
        idx_tr, idx_va = train_test_split(idx_resto, test_size=0.15, stratify=y[idx_resto], random_state=SEMILLA)
        modelo = crear_cnn(TAM)
        modelo.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
        modelo.fit(X[idx_tr, ..., None], y[idx_tr], validation_data=(X[idx_va, ..., None], y[idx_va]),
                   epochs=EPOCAS, batch_size=64, callbacks=callbacks(), verbose=0)
        pred = modelo.predict(X[prueba, ..., None], verbose=0).argmax(axis=1)
        resultados_personas[persona] = (accuracy_score(y[prueba], pred), int(prueba.sum()))
        errores = collections.Counter((CLASES[r], CLASES[p]) for r, p in zip(y[prueba], pred) if r != p)
        errores_personas.update(errores)
        detalle = ", ".join(f"{r}→{p} ×{n}" for (r, p), n in errores.most_common(3)) or "sin errores"
        print(f"Sin {persona:<12} accuracy en sus {prueba.sum()} dibujos: {resultados_personas[persona][0]:.3f}  ({detalle})")
    media = np.mean([acc for acc, _ in resultados_personas.values()])
    print(f"\nAccuracy promedio con personas no vistas: {media:.3f}")
    print("Confusiones más comunes con personas no vistas:",
          ", ".join(f"{r}→{p} ×{n}" for (r, p), n in errores_personas.most_common(5)) or "ninguna")

## Guardar el modelo

In [ ]:
cnn.save("modelo_compuertas.keras")
with open("clases.json", "w") as f:
    json.dump(CLASES, f)
print("Guardado modelo_compuertas.keras y clases.json")

### Exportar el modelo para la app ("Adivina tu dibujo")

La página `/probar` de la app hace la predicción en el navegador con JavaScript puro, sin TensorFlow. Para eso se exportan los pesos de la CNN a dos archivos:
- `modelo_web.json`: la lista de capas (convolución, max pooling, densa) y dónde están sus pesos.
- `modelo_web.bin`: todos los pesos como números `float32`.

Las capas de aumento de datos y de dropout no se exportan porque solo actúan durante el entrenamiento.

Después hay que copiar los dos archivos a la carpeta `static/modelo/` de la app (en PythonAnywhere: pestaña **Files**, subir y luego **Reload**).

In [ ]:
def exportar_web(modelo, carpeta="modelo_web"):
    os.makedirs(carpeta, exist_ok=True)
    capas, bloques, posicion = [], [], 0

    def agregar(arreglo):
        nonlocal posicion
        plano = np.asarray(arreglo, dtype=np.float32).ravel()
        bloques.append(plano)
        inicio, posicion = posicion, posicion + plano.size
        return [inicio, int(plano.size)]

    for capa in modelo.layers:
        if isinstance(capa, layers.Conv2D):
            conf = capa.get_config()
            assert conf["padding"] == "same" and tuple(conf["strides"]) == (1, 1)
            W, b = capa.get_weights()
            capas.append({"tipo": "conv2d", "kernel": int(W.shape[0]), "filtros": int(W.shape[3]),
                          "activacion": conf["activation"], "w": agregar(W), "b": agregar(b)})
        elif isinstance(capa, layers.MaxPooling2D):
            capas.append({"tipo": "maxpool", "tam": int(capa.get_config()["pool_size"][0])})
        elif isinstance(capa, layers.Flatten):
            capas.append({"tipo": "flatten"})
        elif isinstance(capa, layers.Dense):
            W, b = capa.get_weights()
            capas.append({"tipo": "dense", "entrada": int(W.shape[0]), "unidades": int(W.shape[1]),
                          "activacion": capa.get_config()["activation"], "w": agregar(W), "b": agregar(b)})
        # El aumento de datos y el dropout no se usan al predecir.

    manifiesto = {"tam": int(modelo.input_shape[1]), "margen": MARGEN, "clases": CLASES, "capas": capas}
    with open(os.path.join(carpeta, "modelo_web.json"), "w") as f:
        json.dump(manifiesto, f)
    np.concatenate(bloques).astype("<f4").tofile(os.path.join(carpeta, "modelo_web.bin"))
    print(f"Exportado en {carpeta}/: {len(capas)} capas, {posicion:,} pesos")

exportar_web(cnn)

try:
    from google.colab import files
    import shutil
    shutil.make_archive("modelo_web", "zip", "modelo_web")
    files.download("modelo_web.zip")
except ImportError:
    pass

## Resumen de resultados

Esta celda imprime los números clave para copiarlos en las conclusiones y en la exposición.

In [ ]:
acc_mlp, f1_mlp = resumen[0][1], resumen[0][2]
acc_cnn, f1_cnn = resumen[1][1], resumen[1][2]
print(f"Dataset: {len(y)} dibujos, {len(set(autores))} personas")
print(f"MLP: accuracy {acc_mlp:.1%}, F1 macro {f1_mlp:.3f}")
print(f"CNN: accuracy {acc_cnn:.1%}, F1 macro {f1_cnn:.3f}")
f1_clases = f1_score(y_test, pred_cnn, average=None)
if f1_clases.min() == f1_clases.max():
    print(f"Clase más difícil para la CNN: ninguna, todas con F1 {f1_clases.min():.3f} en prueba")
else:
    peor = int(np.argmin(f1_clases))
    print(f"Clase más difícil para la CNN: {CLASES[peor]} (F1 {f1_clases[peor]:.3f})")
for tam, r in resultados_res.items():
    print(f"CNN a {tam}×{tam}: accuracy {r['accuracy']:.1%}, confusión AND↔NAND {r['and_nand']:.1f}%, OR↔NOR {r['or_nor']:.1f}%")
if "resultados_personas" in globals():
    print(f"CNN con personas no vistas: accuracy promedio {media:.1%}")
    print("Confusiones más comunes con personas no vistas:",
          ", ".join(f"{r}→{p} ×{n}" for (r, p), n in errores_personas.most_common(3)) or "ninguna")

## 11. Conclusiones

*Completar con los números de la celda anterior una vez entrenado con los datos reales.*

1. **CNN contra MLP:** la CNN obtuvo ___% de accuracy frente a ___% del MLP. Las convoluciones aprovechan la estructura espacial del dibujo (bordes, curvas, círculos), mientras que el MLP ve píxeles sueltos.
2. **Parejas difíciles:** la mayor confusión fue ___ ↔ ___ (___%). Coincide con lo esperado: estas compuertas solo se diferencian por ___.
3. **Resolución:** a 16×16 la accuracy fue ___% y a 64×64 ___%. A baja resolución el círculo de NAND/NOR se pierde, lo que muestra que aumentar la dimensión de la imagen ayuda cuando la diferencia entre clases está en detalles pequeños.
4. **Personas nuevas:** con dibujantes que la red no vio, la accuracy bajó a ___%. El estilo personal influye, y más personas aportando datos mejorarían la generalización.
5. **Limitaciones y trabajo futuro:** el sistema clasifica compuertas sueltas; para digitalizar un circuito completo haría falta detectar y separar cada compuerta y sus conexiones (detección de objetos).